# Block 3 — Range, velocity, and ambiguity
### How does the radar know how far and how fast — and where does that break?

Range and velocity both come from the *same* pulse train, and the pulse repetition frequency that helps one hurts the other. Three widgets walk from range folding, to the fixed range–velocity trade-off, to the aliasing you see on a real display. Sliders only; run setup first.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

C = 2.998e8

def lam(freq_ghz):        return C/(freq_ghz*1e9)
def r_max_km(prf):        return C/(2*prf)/1000.0          # max unambiguous range (km)
def v_nyq(prf, f=2.8):    return lam(f)*prf/4.0            # Nyquist velocity (m/s)
def fold_velocity(v, vmax): return ((v + vmax) % (2*vmax)) - vmax
def product_const(f=2.8): return C*lam(f)/8000.0          # the cl/8 ceiling, km*(m/s)

print("Block 3 ready | PRF 1000 Hz -> r_max", round(r_max_km(1000)), "km, Nyquist",
      round(v_nyq(1000),1), "m/s | product", round(r_max_km(1000)*v_nyq(1000)))

Block 3 ready | PRF 1000 Hz -> r_max 150 km, Nyquist 26.8 m/s | product 4013


## 1 — The pulse train sets the rules: range folding

A pulse must return before the next one goes out, or it's misplaced to a shorter range — a **second-trip echo**. The maximum unambiguous range is r_max = c/(2·PRF). Push a distant target past it and watch where it lands.

In [2]:
def plot_folding(prf=1000, pulse_us=1.57, target_km=320):
    rmax = r_max_km(prf); dr = C*pulse_us*1e-6/2; prt = 1000.0/prf
    fig, (a, b) = plt.subplots(2, 1, figsize=(8, 5.4), gridspec_kw=dict(height_ratios=[1, 2]))
    for k in range(4): a.axvline(k*prt, 0.05, 0.95, color="tab:blue", lw=2)
    for k in range(3): a.axvspan(k*prt, (k+1)*prt, color="gray", alpha=0.08)
    a.annotate("", (prt, 1.15), (0, 1.15), arrowprops=dict(arrowstyle="<->"))
    a.text(prt/2, 1.25, "PRT = " + str(round(prt,2)) + " ms", ha="center", fontsize=9)
    a.set_xlim(-0.05*prt, 3*prt); a.set_ylim(0, 1.5); a.set_yticks([]); a.set_xlabel("time (ms)")
    a.set_title("pulse train @ " + str(int(prf)) + " Hz  |  r_max = " + str(round(rmax)) + " km  |  dr = " + str(round(dr)) + " m")
    r = np.linspace(0, 500, 1000)
    b.plot(r, r % rmax, color="tab:blue", lw=2, label="displayed range")
    b.plot(r, r, color="gray", ls="--", lw=1, label="true = displayed")
    b.axvline(rmax, color="tab:red", ls=":")
    dt = target_km % rmax; b.plot([target_km], [dt], "o", color="tab:red", ms=8)
    if target_km > rmax:
        b.annotate("true " + str(int(target_km)) + " km shown at " + str(round(dt)) + " km",
                   (target_km, dt), (40, dt+150), fontsize=8, color="tab:red", arrowprops=dict(arrowstyle="->"))
    b.set_xlim(0, 500); b.set_ylim(0, 560); b.set_xlabel("true range (km)"); b.set_ylabel("displayed range (km)")
    b.legend(fontsize=8, loc="upper right"); b.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(plot_folding,
         prf=IntSlider(value=1000, min=250, max=1300, step=10, description="PRF Hz"),
         pulse_us=FloatSlider(value=1.57, min=0.5, max=5, step=0.1, description="pulse µs"),
         target_km=IntSlider(value=320, min=20, max=480, step=10, description="target km"));

interactive(children=(IntSlider(value=1000, description='PRF Hz', max=1300, min=250, step=10), FloatSlider(val…

**Basic**

1. At 1000 Hz, what is the maximum unambiguous range? Send a target to 320 km — where does it appear?
2. Lower the PRF. Does r_max grow or shrink?

**A little further**

1. Set a PRF so a storm at 300 km is *not* folded. (Peek at the next widget) what Nyquist velocity did that cost you?
2. A second-trip echo shows up at (true range − r_max). Reason from the pulse timing why that's exactly where it lands.

## 2 — The trade you can't escape: the Doppler dilemma

Raising the PRF buys Nyquist velocity but spends range; lowering it does the reverse. Their product r_max·v_max = cλ/8 is fixed by wavelength alone, so you only ever slide along one curve.

In [3]:
def plot_dilemma(prf=1000, band="S"):
    fghz = 2.8 if band == "S" else 5.6
    Ks, Kc = product_const(2.8), product_const(5.6); r = np.linspace(60, 500, 400)
    plt.figure(figsize=(8, 5))
    plt.fill_between(r, Ks/r, 60, color="tab:red", alpha=0.07)
    plt.plot(r, Ks/r, color="tab:blue", lw=2.5, label="S-band")
    plt.plot(r, Kc/r, color="tab:orange", lw=1.75, ls="--", label="C-band")
    rp, vp = r_max_km(prf), v_nyq(prf, fghz)
    plt.plot([rp], [vp], "o", color="tab:blue" if band == "S" else "tab:orange", ms=9)
    plt.annotate("PRF " + str(int(prf)) + " Hz: " + str(round(rp)) + " km, " + str(round(vp)) + " m/s",
                 (rp, vp), (rp+15, vp+6), fontsize=8, arrowprops=dict(arrowstyle="->"))
    plt.plot([150], [35], "x", color="tab:red", ms=11, mew=3)
    plt.text(150, 38, "storm 35 m/s @ 150 km", color="tab:red", fontsize=8, ha="center")
    plt.xlim(0, 500); plt.ylim(0, 60); plt.grid(alpha=0.3); plt.legend()
    plt.xlabel("max unambiguous range (km)"); plt.ylabel("Nyquist velocity (m/s)")
    plt.title("the Doppler dilemma: you ride a fixed curve")
    plt.show()

interact(plot_dilemma,
         prf=IntSlider(value=1000, min=250, max=1300, step=10, description="PRF Hz"),
         band=Dropdown(options=["S", "C"], value="S", description="band"));

interactive(children=(IntSlider(value=1000, description='PRF Hz', max=1300, min=250, step=10), Dropdown(descri…

**Basic**

1. Slide the PRF. Does the operating point ever leave the curve?
2. Read r_max and v_max at 1000 Hz and multiply them. Does the product change as you slide?

**A little further**

1. Can you reach the marked storm (35 m/s at 150 km) at *any* single PRF? Why not, and by roughly what factor are you short?
2. Switch to C-band. Does the dilemma get easier or harder, and by about what factor? (Think cλ/8.)

## 3 — The artifact you'll actually see: velocity aliasing

When a storm moves faster than the Nyquist velocity, its measured velocity **folds** into the ±v_max window (green band) — a fast outbound storm can read as inbound. On a real display that's a sharp, unphysical jump.

In [4]:
def plot_aliasing(prf=1000, true_v=35, band="S"):
    fghz = 2.8 if band == "S" else 5.6; vmax = v_nyq(prf, fghz); vt = np.linspace(-70, 70, 1400)
    plt.figure(figsize=(8, 4.6))
    plt.axhspan(-vmax, vmax, color="tab:green", alpha=0.08)
    plt.plot(vt, fold_velocity(vt, vmax), color="tab:blue", lw=2, label="displayed")
    plt.plot(vt, vt, color="gray", ls="--", lw=1, label="true = displayed")
    d = fold_velocity(true_v, vmax); plt.plot([true_v], [d], "o", color="tab:red", ms=9)
    plt.annotate("true " + str(int(true_v)) + " -> shown " + str(round(d,1)) + ("  ALIASED" if abs(true_v) > vmax else ""),
                 (true_v, d), (true_v-5, d-24), fontsize=9, color="tab:red", arrowprops=dict(arrowstyle="->"))
    plt.xlim(-70, 70); plt.ylim(-70, 70); plt.grid(alpha=0.3); plt.legend(loc="upper left")
    plt.xlabel("true radial velocity (m/s)"); plt.ylabel("displayed velocity (m/s)")
    plt.title("velocity folding  |  Nyquist +/- " + str(round(vmax)) + " m/s @ " + str(int(prf)) + " Hz")
    plt.show()

interact(plot_aliasing,
         prf=IntSlider(value=1000, min=250, max=1300, step=10, description="PRF Hz"),
         true_v=FloatSlider(value=35, min=-70, max=70, step=1, description="true v"),
         band=Dropdown(options=["S", "C"], value="S", description="band"));

interactive(children=(IntSlider(value=1000, description='PRF Hz', max=1300, min=250, step=10), FloatSlider(val…

**Basic**

1. At 1000 Hz, what is the Nyquist velocity? Set a 35 m/s storm — what velocity is displayed?
2. Raise the PRF until 35 m/s is no longer aliased. What range did you give up (recall widget 1)?

**A little further**

1. A display shows a sudden jump from +25 to −25 m/s across a smooth gradient. Real wind shear, or aliasing? Use the widget to make the argument.
2. Dual-PRF can lift the Nyquist ~2–4×. Reasoning from the dilemma, would that let you keep 150 km range *and* capture the 35 m/s storm?

## Facilitator notes
- This block is the time-domain counterpart to Block 1's geometry: same radar, sampled in time, with the ambiguities that follow.
- The one idea to leave with: *range and velocity share one pulse train, and r_max·v_max = cλ/8 is a wall — a single scan gives long range or high Nyquist, not both.*
- Ties forward to Block 4: it's why NEXRAD splits its low-tilt scan in two, and why C-band networks lean on staggered-PRT.